# Part A (PM) — 8-Algorithm ML Cheat Sheet & Comparison
**Day 33 | PM Session | Week 6**

For each of the 8 algorithms learned this week we build an 'algorithm card' (when to use, key params, pros, cons, 5-line code), then run all 8 on a single dataset with proper cross-validation and rank them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# --- Algorithm imports ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed; substituting GradientBoostingClassifier as Algorithm 8.')

print('Imports OK.')

---
## Algorithm Cards

In [ ]:
CARDS = [
    {
        'name': 'Logistic Regression',
        'when': 'Linearly separable data, need probability output, large dataset.',
        'params': 'C (inverse regularisation), solver, max_iter, penalty (l1/l2)',
        'pros': ['Fast training', 'Interpretable (log-odds)', 'Calibrated probabilities', 'Works well on high-dim text'],
        'cons': ['Assumes linear boundary', 'Sensitive to outliers without regularisation'],
        'code': """
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
model = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(C=1.0, max_iter=1000))])
model.fit(X_train, y_train); print(model.score(X_test, y_test))"""
    },
    {
        'name': 'Decision Tree',
        'when': 'Need interpretability, mixed feature types, no scaling required.',
        'params': 'max_depth, min_samples_split, min_samples_leaf, criterion',
        'pros': ['Interpretable (tree plot)', 'No scaling', 'Handles non-linearity'],
        'cons': ['High variance (overfits easily)', 'Unstable — small data changes → different tree'],
        'code': """
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))"""
    },
    {
        'name': 'Random Forest',
        'when': 'General-purpose, tabular data, need feature importance, robust to overfitting.',
        'params': 'n_estimators, max_depth, max_features, min_samples_leaf',
        'pros': ['Low variance via bagging', 'Built-in feature importance', 'Parallelisable'],
        'cons': ['Less interpretable than single tree', 'Slow on very high-dim data'],
        'code': """
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))"""
    },
    {
        'name': 'Gradient Boosting',
        'when': 'Best accuracy on tabular data, can handle heterogeneous features.',
        'params': 'n_estimators, learning_rate, max_depth, subsample',
        'pros': ['Often best accuracy', 'Handles missing values (XGB)', 'Regularised'],
        'cons': ['Slow to train', 'Many hyperparameters', 'Prone to overfit without tuning'],
        'code': """
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))"""
    },
    {
        'name': 'SVM (RBF)',
        'when': 'High-dimensional data, small-medium dataset, non-linear boundary needed.',
        'params': 'C, gamma, kernel (rbf/poly/linear)',
        'pros': ['Effective in high dim', 'Robust to outliers (only SVs matter)', 'Kernel trick for non-linearity'],
        'cons': ['Does not scale to large n', 'Slow to tune', 'No probability output by default'],
        'code': """
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
model = Pipeline([('sc', StandardScaler()), ('svm', SVC(kernel='rbf', C=10, gamma=0.01))])
model.fit(X_train, y_train); print(model.score(X_test, y_test))"""
    },
    {
        'name': 'KNN',
        'when': 'Small dataset, instance-based learning, quick baseline, similarity-based tasks.',
        'params': 'n_neighbors (K), metric (euclidean/manhattan/minkowski), weights',
        'pros': ['No training phase', 'Naturally multi-class', 'Adapts to local structure'],
        'cons': ['Slow at inference O(nd)', 'Curse of dimensionality', 'Sensitive to irrelevant features'],
        'code': """
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
model = Pipeline([('sc', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
model.fit(X_train, y_train); print(model.score(X_test, y_test))"""
    },
    {
        'name': 'Naive Bayes (Gaussian)',
        'when': 'Text/document classification, very small data, need fast probabilistic baseline.',
        'params': 'var_smoothing (prevents zero probabilities)',
        'pros': ['Extremely fast', 'Works well with small data', 'Probabilistic output'],
        'cons': ['Strong independence assumption (rarely true)', 'Underperforms when features are correlated'],
        'code': """
from sklearn.naive_bayes import GaussianNB
model = GaussianNB(var_smoothing=1e-9)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))"""
    },
    {
        'name': 'XGBoost',
        'when': 'Tabular data, need best accuracy, kaggle competitions, handles missing values.',
        'params': 'n_estimators, learning_rate, max_depth, subsample, colsample_bytree, reg_alpha/lambda',
        'pros': ['Fast (C++ backend)', 'Handles missing values natively', 'Regularised boosting'],
        'cons': ['Less interpretable', 'Many hyperparameters', 'Not ideal for very high-dim sparse data'],
        'code': """
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                      eval_metric='logloss', random_state=42)
model.fit(X_train, y_train); print(model.score(X_test, y_test))"""
    },
]

# Pretty-print cards
for card in CARDS:
    print('='*60)
    print(f"=== {card['name']} ===")
    print(f"When   : {card['when']}")
    print(f"Params : {card['params']}")
    print(f"Pros   : {', '.join(card['pros'])}")
    print(f"Cons   : {', '.join(card['cons'])}")
    print(f"Code   :{card['code']}")
print('='*60)

## Run All 8 on Breast Cancer Dataset (same CV, same scaling)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print(f'Dataset: Breast Cancer | shape: {X.shape} | classes: {np.unique(y)}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Models — all wrapped in Pipelines with StandardScaler where needed
models = {
    'Logistic Regression'  : Pipeline([('sc', StandardScaler()), ('m', LogisticRegression(C=1.0, max_iter=1000))]),
    'Decision Tree'        : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'        : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting'    : GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
    'SVM (RBF)'            : Pipeline([('sc', StandardScaler()), ('m', SVC(kernel='rbf', C=10, gamma='scale'))]),
    'KNN'                  : Pipeline([('sc', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))]),
    'Naive Bayes'          : GaussianNB(),
}

if HAS_XGB:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                       eval_metric='logloss', random_state=42, verbosity=0)
else:
    models['GradBoost-2'] = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    results[name] = scores
    print(f'{name:<25}  mean={scores.mean():.4f}  std={scores.std():.4f}')

In [ ]:
# Visualise results
res_df = pd.DataFrame(results).T
res_df.columns = [f'Fold {i+1}' for i in range(5)]
res_df['Mean'] = res_df.mean(axis=1)
res_df['Std']  = res_df.iloc[:, :5].std(axis=1)
res_df = res_df.sort_values('Mean', ascending=False)

print('\n--- Ranked Results ---')
print(res_df[['Mean', 'Std']].to_string())

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(res_df)))
bars = ax.barh(res_df.index[::-1], res_df['Mean'][::-1], xerr=res_df['Std'][::-1],
               color=colors, edgecolor='k', linewidth=0.5, capsize=4)
ax.set_xlabel('5-Fold CV Accuracy', fontsize=12)
ax.set_title('Algorithm Comparison — Breast Cancer Dataset', fontsize=13)
ax.set_xlim(0.88, 1.0)
ax.bar_label(bars, labels=[f'{v:.4f}' for v in res_df['Mean'][::-1]], padding=4, fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Recommendation

In [ ]:
best_model = res_df['Mean'].idxmax()
best_score = res_df['Mean'].max()

print(f"Best model  : {best_model}")
print(f"CV accuracy : {best_score:.4f}")
print()
print("Recommendation reasoning:")
print("  - Ensemble methods (RF, GBM, XGB) consistently top the leaderboard on tabular data.")
print("  - SVM(RBF) is competitive and excellent when data is moderate-sized (<50K rows).")
print("  - Logistic Regression is a strong baseline and most interpretable.")
print("  - Decision Tree and KNN are weaker here due to high dimensionality (30 features).")
print("  - Naive Bayes is fastest but assumes feature independence — less accurate here.")
print()
print("For production on this dataset: use GBM/XGB for accuracy OR Logistic Regression")
print("for interpretability (medical domain often requires explainability).")